# Experiment 2 — Temporal Trajectory Features vs. Static Snapshot

Compares the predictive power of temporal biomarker trajectory features
(slope, velocity, exponential smoothing, moving average) against a static
last-value snapshot for cancer triage.

**Key outputs:**
- AUROC delta: temporal − static
- Feature importance by trajectory statistic type
- Grouped bar chart: static vs temporal AUROC per cancer type

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.seeding import seed_everything
from src.features.feature_pipeline import CancerTriageFeaturePipeline, load_horizon_data, create_train_val_test_split
from src.models.temporal_model import TemporalModelEvaluator

seed_everything(42)
sns.set_theme(style='whitegrid', font_scale=1.2)
os.makedirs('../results/figures', exist_ok=True)
print('Ready.')

In [ ]:
cfg = {
    'seed': 42,
    'paths': {'data_processed': '../data/processed', 'figures': '../results/figures'},
    'features': {'cbc': ['wbc','hemoglobin','platelets','neutrophils','lymphocytes'],
                 'metabolic': ['glucose','albumin','creatinine'],
                 'inflammatory': ['crp']}
}

X, y = load_horizon_data('../data/processed', horizon_months=12, dataset='mimic')
X_train, X_val, X_test, y_train, y_val, y_test = create_train_val_test_split(X, y)

pipeline = CancerTriageFeaturePipeline(cfg)
X_train_p = pipeline.fit_transform(X_train, y_train)
X_test_p  = pipeline.transform(X_test)
print('Data loaded.')

In [ ]:
evaluator = TemporalModelEvaluator(cfg)
comparison_df = evaluator.compare_static_vs_temporal(
    X_static_train=X_train_p,
    X_static_test=X_test_p,
    X_temporal_train=X_train_p,
    X_temporal_test=X_test_p,
    y_train=y_train,
    y_test=y_test
)
print(comparison_df.to_string())

In [ ]:
evaluator.plot_temporal_comparison(comparison_df, '../results/figures/exp2_temporal_vs_static.png')
comparison_df.to_csv('../results/exp2_temporal_vs_static.csv')
print('Saved.')